# 01 - Look at the data + simple baseline

**Challenge:** predict which passengers were **transported** to another dimension (`True`/`False`).
**Score:** accuracy. **Submit:** `PassengerId, Transported` for 4,277 test passengers.

This notebook: explore the data, then build a simple logistic regression → `submission_v1.csv`.

In [1]:
# --- Libraries ---
import warnings
warnings.filterwarnings("ignore")                     # hide harmless library warnings
import numpy as np                                    # maths helpers
import pandas as pd                                   # tables (DataFrames)
from sklearn.linear_model import LogisticRegression   # a simple yes/no model
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score   # tools to check accuracy
from sklearn.pipeline import make_pipeline            # chain steps: scale data -> model
from sklearn.preprocessing import StandardScaler      # puts numbers on the same scale

# --- Load the data ---
train = pd.read_csv("spaceship-titanic-data/train.csv")   # has the answer column "Transported"
test = pd.read_csv("spaceship-titanic-data/test.csv")     # no answers - we predict these

# Stack train and test into one table, so cleaning steps are applied the same way to both
df = pd.concat([train, test], ignore_index=True)
is_train = df.Transported.notna()                     # True for train rows, False for test rows

# The five money columns (how much each passenger spent on board)
SPEND = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

# PassengerId looks like "0013_01": first 4 digits = travel group, last 2 = person in that group
df["Group"] = df.PassengerId.str[:4]

y = train.Transported.astype(int)                     # the answers as 1 (transported) / 0 (not)
groups = df.loc[is_train, "Group"]                    # group of each train passenger

# How we check accuracy: split train into 5 parts, train on 4, test on the 5th, repeat 5 times.
# "Group" version keeps a whole travel group in the same part. Otherwise the model could
# learn from someone's family and be tested on them, which makes the score look too good.
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)

## Quick look

In [2]:
# How big is the data, and how many passengers were transported?
print("Train rows:", len(train), "| Test rows:", len(test))
print("Transported rate:", round(train.Transported.mean(), 3), "(about half, so guessing gets 50%)")

# Show the first 5 rows so we can see what each column looks like
train.head()

Train rows: 8693 | Test rows: 4277
Transported rate: 0.504 (about half, so guessing gets 50%)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


## Missing values
Every column has about 2% missing. These need filling.

In [3]:
# Count empty cells in each column of the train data
train.isna().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

## What matters most?

In [4]:
# Did the passenger spend any money at all? (True / False)
spent_any = train[SPEND].sum(axis=1) > 0

# Cabin looks like "B/0/P" = deck / number / side. Split it into 3 parts
cabin = train.Cabin.str.split("/", expand=True)

# For each column, show the share of passengers transported in each category
for name, col in [("CryoSleep", train.CryoSleep), ("HomePlanet", train.HomePlanet),
                  ("Spent any money", spent_any), ("Cabin deck", cabin[0]), ("Cabin side", cabin[2])]:
    print(f"--- {name}")
    print(train.groupby(col).Transported.mean().round(2).to_string(), "\n")

--- CryoSleep
CryoSleep
False    0.33
True     0.82 

--- HomePlanet
HomePlanet
Earth     0.42
Europa    0.66
Mars      0.52 

--- Spent any money
False    0.79
True     0.30 

--- Cabin deck
0
A    0.50
B    0.73
C    0.68
D    0.43
E    0.36
F    0.44
G    0.52
T    0.20 

--- Cabin side
2
P    0.45
S    0.56 



**Main signals**
- **CryoSleep:** 82% of sleeping passengers were transported vs 33% awake.
- **Spending:** people who spent nothing were transported far more (79% vs 30%). Sleepers can't spend, so these two are linked.
- **Home planet and cabin deck/side** also matter.

## Travel groups
`PassengerId` = `group_number`. Groups never appear in both train and test, so we can't copy answers between them like in the Titanic family rule.
But we must keep each group inside one fold when checking accuracy.

In [5]:
# Take the group number of every test passenger and check if that group is in train
test_groups = test.PassengerId.str[:4]
print("Test passengers whose group is also in train:", test_groups.isin(groups).sum())

Test passengers whose group is also in train: 0


## Baseline model
Simple fills: spending → 0, age → median, missing yes/no → False.

In [6]:
def simple_features(d):
    """Turn raw columns into numbers a model can use, with simple gap filling."""
    X = pd.DataFrame(index=d.index)

    # Money columns: a missing value most likely means "spent nothing"
    for c in SPEND:
        X[c] = d[c].fillna(0)

    # Missing age -> the middle (median) age
    X["Age"] = d.Age.fillna(d.Age.median())

    # Yes/no columns: missing -> False, then True/False -> 1/0
    X["CryoSleep"] = d.CryoSleep.fillna(False).astype(int)
    X["VIP"] = d.VIP.fillna(False).astype(int)

    # Text columns -> one 0/1 column per value (e.g. HomePlanet_Earth, HomePlanet_Mars ...)
    for c in ["HomePlanet", "Destination"]:
        X = X.join(pd.get_dummies(d[c], prefix=c).astype(int))
    return X

# Build the features for everyone, then split back into train and test
X_all = simple_features(df)
X, X_test = X_all[is_train], X_all[~is_train]

# Model = scale the numbers, then logistic regression
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

# Check accuracy with the 5-part split defined at the top
scores = cross_val_score(model, X, y, cv=cv, groups=groups)
print(f"Logistic regression (5-fold check): {scores.mean():.3f} (+/- {scores.std():.3f})")

Logistic regression (5-fold check): 0.786 (+/- 0.010)


## Write submission v1
`Transported` must be `True`/`False`, not 1/0.

In [7]:
# Train the model on ALL train data, then predict the test passengers
model.fit(X, y)

# Kaggle wants True/False, so convert the 1/0 predictions with astype(bool)
sub = pd.DataFrame({"PassengerId": test.PassengerId, "Transported": model.predict(X_test).astype(bool)})
sub.to_csv("submission_v1.csv", index=False)     # index=False: don't write row numbers
print(len(sub), "rows")
sub.head()

4277 rows


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,False


## Result
- Check on train: about **0.786**.
- Next (notebook 02): smarter gap filling, more features, and tree models.